### Virtual-ISP Learning Notebook

This notebook guides you through exploring, preprocessing, modeling, and evaluating data in the Virtual-ISP project. You can experiment with RAW image processing and basic machine learning concepts step by step.

### Import Required Libraries
Import essential libraries for image processing and machine learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rawpy

## Rawpy

A raw image contain direct sensor data from your camera with minimal or no processing. This data is not a standard image and it's a grid of sensor values, usually arranged in a Bayer pattern, and doesn't look like a normal photo if you try to display it directly. 

Rawpy offers a method **postprocess()** which converts the sensor data into a standard RGB image. This step includes demosaicing (i.e. turning the Bayer pattern into color pixels), applying white balance, and some basic color correction similar to what a camera does when creating a JPEG. 

### Load and Explore the Dataset
Load a RAW (ARW) image 

In [ ]:
def display_arw_image(file_path):
    with rawpy.imread(file_path) as raw:
        rgb = raw.postprocess() 
    plt.imshow(rgb)
    plt.title('ARW Image')
    plt.axis('off')
    plt.show()

display_arw_image('.\imgs\AKG02229.ARW') # test image

# Linear
Each pixel is a sensor intensity fraction that we set between 0-1. If a value reads ~0.0 that pixel is at/under black level (no useful light signal). 
If the value reads ~1, that pixel is at/over sensor white level (saturated). 
Values in the middle like .25 or .6 are proportional light measurements. 

We calculate linear by accepting the bayer, black & white level from the decoded sensor values. 
Then we calculate by subtracting the bayer by the black scalar... dividing the result by the difference of white level and black scalar. 

We use np.clip to limit the values between 0 and 1. 

In [2]:
import numpy as np
import rawpy
import matplotlib.pyplot as plt

In [3]:
def decode_arw_image(file_path):
    with rawpy.imread(file_path) as raw:
        bayer = raw.raw_image_visible.copy()  # 2D Bayer mosaic (decoded RAW data)
        cfa = raw.raw_pattern.copy()  # CFA layout, [[0,1], [1,2]]
        black = np.array(raw.black_level_per_channel)
        white = raw.white_level
        color_desc = raw.color_desc

    return bayer, cfa, black, white, color_desc

def linearize_bayer(bayer, black_level, white_level):
    black_scalar = float(black_level[0])
    linear = (bayer.astype(np.float32) - black_scalar) / \
        (white_level - black_scalar)
    return np.clip(linear, 0.0, 1.0) # limit linear values between 0 and 1



bayer, cfa, black, white, color_desc = decode_arw_image('.\imgs\AKG02229.ARW')
linear = linearize_bayer(bayer, black, white) # each pixel is a sensor intensity fraction

print(linear)

<>:19: SyntaxWarning: invalid escape sequence '\i'
<>:19: SyntaxWarning: invalid escape sequence '\i'
C:\Users\brandon.hernandez\AppData\Local\Temp\ipykernel_84360\2116449761.py:19: SyntaxWarning: invalid escape sequence '\i'
  bayer, cfa, black, white, color_desc = decode_arw_image('.\imgs\AKG02229.ARW')


[[0.01310566 0.03994707 0.01121542 ... 1.         0.52680993 1.        ]
 [0.03799383 0.01587802 0.03767879 ... 1.         1.         1.        ]
 [0.01461786 0.03938    0.01291664 ... 1.         0.5448932  1.        ]
 ...
 [0.07976813 0.04473568 0.08808519 ... 0.01096339 0.0212337  0.01159347]
 [0.03452839 0.07983114 0.03730074 ... 0.02066663 0.00762397 0.02072963]
 [0.07327831 0.04505072 0.08770714 ... 0.01190851 0.0189024  0.01146746]]


### Use the CFA as a repeating “label map” over the full Bayer image:

raw_pattern is a 2x2 tile of channel IDs.

That 2x2 tile repeats across the whole sensor.

Each ID maps to a color via color_desc (often like RGBG, so two IDs can both be green).

Build boolean masks for R/G/B, then pull values from linear into channel planes.

### How to apply a CFA pattern across the whole image

In [ ]:
h, w = bayer.shape

full_ids = np.tile(cfa, (h // 2 + 1, w // 2 + 1))[:h, :w]


Here we are using tile from numpy which will repeat a pattern across the sensor (bayer). We divide the height and width by 2 + 1 to provide a tile of safety. That is why we crop from the beginning to the extent of height and width. 